# Saltback simulator - Steps 1 and 2
**How to use:** click **Runtime -> Run all**. Then move the sliders in step 2 and re-run steps 2-4.

- **Step 1** works out how much ice a salt dose *can* melt at a given road temperature.
- **Step 2** plays the storm in 5-minute steps: snow, plow-and-salt passes, salt dissolving, brine melting
  or refreezing into black ice, runoff into the storm drain, and the probe's live reading.

The simulation (`simulate.py`) only produces raw numbers. A separate translator (`explain.py`)
turns them into plain English and a chart. All inputs are typical published values, not real town data.

In [ ]:
# @title 1. Load the Saltback model (click to run; no need to read the code)
open('model.py', 'w').write(r'''"""Saltback physics model.
  STEP 1  storm()          one storm, all at once: how much ice a salt dose CAN melt (equilibrium).
  STEP 2  storm_timeline() the same storm minute by minute: snowfall, plowing, salt dissolving,
                           brine melting or refreezing, runoff reaching the drain, the sensor's live trace.

Follows the salt along its whole path:
    truck -> road (melts ice) -> runoff -> storm drain -> sensor reading -> Saltback's estimate

Everything is "per square meter of road", so results do not depend on road size.
SETTINGS holds typical published values (ASSUMPTIONS, not measured data); change them freely.
"""
import math
import numpy as np

# ---------------- Settings you can change (typical values, with realistic ranges) ----------------
SETTINGS = dict(
    pavement_temp_F=22,       # road surface temperature during the storm               (range -5 to 32)
    ice_film_mm=0.2,          # ice film left after plowing, bonded to pavement, mm water (range 0.05 to 0.5)
    scatter_loss=0.20,        # share of salt that bounces/blows off the road            (0.05 wet - 0.30 dry)
    later_water_mm=5.0,       # extra snowmelt/rain that later rinses the road, mm        (range 0 to 20)
    unsalted_share=0.30,      # share of drain water from unsalted surfaces (lawns, roofs) (range 0 to 0.7)
    baseline_cl_mg_L=50,      # chloride already in drain water before the storm          (range 10 to 500)
    water_temp_C=1.0,         # temperature of the runoff where the probe sits            (range 0 to 5)
    probe_max_mS_cm=20.0,     # highest conductivity the DFRobot DFR0300 (K=1) can read
    probe_noise=0.03,         # random probe error, as a fraction of the reading (3%)
    temp_probe_error_C=0.5,   # DS18B20 temperature probe accuracy
    use_temp_probe=True,      # False = Saltback skips the cold-water correction (to show why it matters)
    salt_price_per_ton=90,    # US$ per short ton of road salt, delivered                 (range 60 to 130)
    seed=1,                   # fixes the random sensor noise so runs are repeatable
    # --- Step 2: the storm over time ---
    storm_hours=8,            # how long it snows                                          (2 to 24)
    snowfall_in=4,            # total snowfall, inches                                     (0 to 18)
    snow_ratio=12,            # inches of snow per inch of water (wet snow 8, fluffy 18)
    first_pass_h=0.5,         # hours of snow before the first truck arrives              (0 to 3)
    plow_every_h=2,           # then a truck plows AND salts every N hours while it snows (1 to 4)
    salt_per_pass_lb=220,     # salt spread each pass, lb per lane-mile                   (0 to 500)
    temp_after_F=38,          # road temperature once the storm is over (sun/thaw)        (10 to 50)
    drain_per_h=0.3,          # share of liquid brine that runs off the road each hour    (0.1 to 0.8)
    traffic_loss_per_h=0.05,  # share of brine/salt that tires throw onto the shoulder /h (0.02 to 0.15)
    bond_per_h=0.5,           # how fast traffic packs unprotected snow into bonded ice /h (0.2 to 1)
    brine_film_mm=0.04,       # brine layer that stops snow bonding to pavement, mm (0.02 to 0.06; tuned to state rate charts)
    catchment_m2=400,         # road area draining to the monitored storm drain           (100 to 2000)
    route_lane_miles=10,      # size of the route, for the storm card                     (1 to 50)
    cleanup_h=0,              # after the storm, trucks keep coming back on schedule for up to this many hours
                              # while the road is below freezing and not yet clear (0 = off; the many-winters
                              # simulator uses 12)                                        (0 to 24)
)

# ---------------- Fixed science (reference-table values, not assumptions) ----------------
CL_SHARE = 35.45 / 58.44                       # chloride is 60.7% of salt (NaCl) by mass
LANE_MILE_M2 = 1609.34 * 3.658                 # one 12-ft lane, one mile long = 5,887 m^2
LB_PER_LANE_MILE = LANE_MILE_M2 / 453.59       # 1 g/m^2 = 13.0 lb per lane-mile

# Freezing point of salt water vs. salt strength (CRC Handbook, NaCl). Below -21.1 C salt cannot melt ice.
BRINE_PCT = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16, 18, 20, 22, 23.3]
FREEZE_C = [0, -0.59, -1.19, -1.79, -2.41, -3.05, -3.70, -4.38, -5.08, -5.81,
            -6.56, -8.18, -9.94, -11.89, -14.04, -16.46, -19.18, -21.1]

# Conductivity of salt water at 25 C (mS/cm) vs. salt concentration (g NaCl per liter). Standard values.
NACL_G_L = [0, 0.584, 2.92, 5.84, 29.2, 58.4, 117, 234, 317]
EC25 = [0, 1.185, 5.55, 10.67, 47.0, 85.8, 145, 214, 251]
ALPHA_TRUE = 0.021    # real NaCl conductivity change per C (about 2.1% per degree)
ALPHA_ASSUMED = 0.02  # the standard 2%/C correction Saltback applies


def f_to_c(f):
    return (np.asarray(f, float) - 32) * 5 / 9


def brine_strength_needed(temp_c):
    """Salt fraction a brine must have to stay liquid at temp_c (0.0 above freezing, nan below -21.1 C)."""
    t = np.asarray(temp_c, float)
    w = np.interp(-t, -np.array(FREEZE_C), BRINE_PCT) / 100
    return np.where(t >= 0, 0.0, np.where(t < FREEZE_C[-1], np.nan, w))


def ice_melted_per_salt(temp_c):
    """Pounds of ice one pound of salt can melt at temp_c (once fully dissolved)."""
    w = brine_strength_needed(temp_c)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(np.isnan(w), 0.0, (1 - w) / w)


def chloride_to_conductivity(cl_mg_L, water_c, alpha=ALPHA_TRUE):
    nacl_g_L = np.asarray(cl_mg_L, float) / CL_SHARE / 1000
    return np.interp(nacl_g_L, NACL_G_L, EC25) * (1 + alpha * (water_c - 25))


def conductivity_to_chloride(ec, water_c, alpha=ALPHA_ASSUMED):
    ec25 = np.asarray(ec, float) / (1 + alpha * (water_c - 25))
    return np.interp(ec25, EC25, NACL_G_L) * CL_SHARE * 1000


def sensor(cl_true, s, rng):
    """What the DFR0300 + DS18B20 report, and the chloride Saltback works out from that reading."""
    ec = chloride_to_conductivity(cl_true, s["water_temp_C"])
    reading = ec * (1 + s["probe_noise"] * rng.standard_normal(np.shape(ec)))
    clipped = reading >= s["probe_max_mS_cm"]
    reading = np.minimum(reading, s["probe_max_mS_cm"])
    temp_read = s["water_temp_C"] + s["temp_probe_error_C"] * rng.standard_normal(np.shape(ec))
    cl_est = conductivity_to_chloride(reading, temp_read)
    cl_no_tempfix = conductivity_to_chloride(reading, 25.0)   # what you'd get with no temperature probe
    if not s.get("use_temp_probe", True):
        cl_est = cl_no_tempfix
    return reading, clipped, cl_est, cl_no_tempfix


def storm(dose_g_m2, s, rng):
    """Simulate one storm for each salt dose (g of salt per m^2 of road). Returns a dict of result columns."""
    dose = np.asarray(dose_g_m2, float)
    temp_c = f_to_c(s["pavement_temp_F"])
    ice = s["ice_film_mm"] * 1000                     # g of ice per m^2 (1 mm of water = 1 kg/m^2)
    w = brine_strength_needed(temp_c)

    # --- Road: how much ice the salt can melt ---
    salt_on_road = dose * (1 - s["scatter_loss"])
    if temp_c >= 0:      # above freezing the ice melts by itself
        melted = np.full_like(dose, ice)
    else:
        melted = np.minimum(ice, salt_on_road * ice_melted_per_salt(temp_c))
    if np.isnan(w):      # too cold for salt to work at all: every grain is wasted
        needed, excess = np.inf, dose
    else:
        needed = ice * w / (1 - w) / (1 - s["scatter_loss"]) if w > 0 else 0.0
        excess = np.maximum(dose - needed, 0)

    # --- Brine leaving the road: once the ice is gone, extra salt only makes the brine saltier ---
    brine_w = np.where(salt_on_road <= 0, 0.0,
                       np.where(melted < ice, w, salt_on_road / (salt_on_road + ice)))
    brine_w = np.nan_to_num(np.minimum(brine_w, 0.264))      # 26.4% = fully saturated salt water
    brine_cl = brine_w * (1 + 0.72 * brine_w) * 1e6 * CL_SHARE   # mg chloride per liter of brine

    # --- Storm drain: first rush of brine (peak) and the whole-storm average ---
    mix = 1 - s["unsalted_share"]
    peak_cl = brine_cl * mix + s["baseline_cl_mg_L"]
    water_L = (ice + s["later_water_mm"] * 1000) / 1000 / mix       # liters reaching the drain per m^2 of road
    avg_cl = salt_on_road * CL_SHARE * 1000 / water_L + s["baseline_cl_mg_L"]

    # --- Sensor + Saltback's estimate ---
    pk_read, pk_clip, pk_est, _ = sensor(peak_cl, s, rng)
    av_read, av_clip, av_est, av_no_fix = sensor(avg_cl, s, rng)
    salt_seen = np.maximum(av_est - s["baseline_cl_mg_L"], 0) * water_L / CL_SHARE / 1000   # g/m^2

    lb = LB_PER_LANE_MILE
    cost_per_lb = s["salt_price_per_ton"] / 2000
    return dict(
        salt_applied_lb_per_lane_mile=dose * lb,
        salt_needed_lb_per_lane_mile=np.full_like(dose, needed * lb),
        excess_pct=np.where(dose > 0, 100 * excess / np.maximum(dose, 1e-9), 0),
        ice_cleared_pct=100 * melted / ice,
        wasted_dollars_per_lane_mile=excess * lb * cost_per_lb,
        chloride_released_lb_per_lane_mile=dose * CL_SHARE * lb,
        peak_cl_true_mg_L=peak_cl, peak_reading_mS_cm=pk_read, peak_probe_maxed_out=pk_clip,
        peak_cl_measured_mg_L=pk_est,
        avg_cl_true_mg_L=avg_cl, avg_reading_mS_cm=av_read, avg_probe_maxed_out=av_clip,
        avg_cl_measured_mg_L=av_est, avg_cl_if_no_temp_probe_mg_L=av_no_fix,
        salt_seen_by_sensor_lb_per_lane_mile=salt_seen * lb,
        salt_reaching_road_lb_per_lane_mile=salt_on_road * lb,
    )


# ======================= STEP 2: the storm over time =======================
DT_H = 5 / 60            # time step: 5 minutes
EUTECTIC_C = FREEZE_C[-1]
PLOW_CLEAR = 0.95        # a plow pass scrapes off 95% of LOOSE snow (it cannot lift ice bonded to pavement)
ICY_G = 50               # bonded ice thicker than 0.05 mm counts as an icy, slippery road
NATURAL_MELT = 300       # g/m^2 per hour per degree C above freezing (pavement thaw rate)
CL_LIMITS = (230, 860)   # US EPA chloride criteria for aquatic life: long-term and short-term (mg/L)


def temp_at(t, s):
    """Pavement temperature (F): steady during the storm, then shifts to temp_after_F over 6 hours."""
    if t < s["storm_hours"]:
        return s["pavement_temp_F"]
    return s["pavement_temp_F"] + (s["temp_after_F"] - s["pavement_temp_F"]) * min(1, (t - s["storm_hours"]) / 6)


def dissolve_minutes(temp_c):
    """Typical time for rock salt to dissolve into wet snow/ice: ~15 min near freezing, hours when very cold."""
    return 5.0 if temp_c >= 0 else 15 * math.exp(-0.12 * temp_c)


def storm_timeline(s, per_pass_lb=None, with_sensor=True):
    """Simulate the storm in 5-minute steps. Returns (rows, summary). All amounts are grams per m^2 of road."""
    per_pass = (s["salt_per_pass_lb"] if per_pass_lb is None else per_pass_lb) / LB_PER_LANE_MILE
    hours = s["storm_hours"] + 24
    snow_per_h = s["snowfall_in"] * 25.4 / s["snow_ratio"] * 1000 / s["storm_hours"]
    every = max(1, round(s["plow_every_h"] / DT_H))
    out_f, spray_f = 1 - math.exp(-s["drain_per_h"] * DT_H), 1 - math.exp(-s["traffic_loss_per_h"] * DT_H)
    mix = 1 - s["unsalted_share"]

    first = round(s["first_pass_h"] / DT_H)
    loose = ice = crystals = water = salt = 0.0          # road state; the storm starts on bare pavement
    applied = roadside = drained = icy_h = 0.0
    rows = []
    for i in range(round(hours / DT_H)):
        t = i * DT_H
        tc = (temp_at(t, s) - 32) * 5 / 9
        storming = t < s["storm_hours"]
        # 1) snow falls (melts on contact if the road is above freezing)
        if storming:
            if tc < 0: loose += snow_per_h * DT_H
            else: water += snow_per_h * DT_H
        # 2) plow + salt pass
        cleanup = (not storming and t < s["storm_hours"] + s.get("cleanup_h", 0) and tc < 0
                   and loose + ice >= ICY_G)
        truck = (storming or cleanup) and i >= first and (i - first) % every == 0
        if truck:
            loose *= 1 - PLOW_CLEAR
            applied += per_pass
            roadside += per_pass * s["scatter_loss"]
            crystals += per_pass * (1 - s["scatter_loss"])
        # 3) salt crystals dissolve, but only into moisture, and slower when cold
        if crystals > 0 and loose + ice + water > 0 and tc > EUTECTIC_C:
            d = crystals * (1 - math.exp(-DT_H * 60 / dissolve_minutes(tc)))
            crystals -= d; salt += d
        # 4) melt or refreeze until the brine is exactly as salty as this temperature requires
        if tc >= 0:
            m = min(loose + ice, NATURAL_MELT * tc * DT_H)
        else:
            w = float(brine_strength_needed(tc))
            target = 0.0 if math.isnan(w) else salt * (1 - w) / w     # liquid water the salt can hold
            m = min(loose + ice, target - water) if water < target else -(water - target)
        if m >= 0:                       # melting: bonded ice (touching the brine) goes first
            from_ice = min(ice, m); ice -= from_ice; loose -= m - from_ice; water += m
        else:                            # refreezing: brine too weak -> black ice
            water += m; ice -= m
        # 5) traffic packs unprotected snow into bonded ice
        if tc < 0 and water < s["brine_film_mm"] * 1000:
            k = loose * (1 - math.exp(-s["bond_per_h"] * DT_H)); loose -= k; ice += k
        # 6) brine runs off to the drain, or tires spray it (and loose crystals) onto the shoulder
        qw, qs = water * out_f, salt * out_f
        sw, ss, sc = water * spray_f, salt * spray_f, crystals * spray_f
        water -= qw + sw; salt -= qs + ss; crystals -= sc
        roadside += ss + sc; drained += qs
        # 7) the storm drain: runoff flow and its chloride level
        flow = qw / 1000 / mix * s["catchment_m2"] / DT_H                          # liters per hour
        if qw > 1e-3:
            f = qs / (qs + qw)
            cl = f * (1 + 0.72 * f) * 1e6 * CL_SHARE * mix + s["baseline_cl_mg_L"]
        else:
            cl = float("nan")                                                       # drain is dry
        icy = tc < 0 and ice >= ICY_G
        icy_h += DT_H if icy else 0
        rows.append(dict(hour=round(t, 3), temp_F=round(temp_at(t, s), 1), truck_pass=truck, loose_snow_mm=loose / 1000,
                         bonded_ice_mm=ice / 1000, brine_mm=water / 1000,
                         brine_salt_pct=100 * salt / (salt + water) if water > 0 else 0.0,
                         salt_crystals_g_m2=crystals, drain_flow_L_h=flow, cl_true_mg_L=cl, icy=icy))

    lb = LB_PER_LANE_MILE
    summary = dict(salt_per_pass_lb=per_pass * lb, salt_applied_lb_per_lane_mile=applied * lb,
                   icy_hours=icy_h, salt_to_drain_pct=100 * drained / max(applied, 1e-9),
                   salt_to_roadside_pct=100 * roadside / max(applied, 1e-9),
                   salt_left_on_road_pct=100 * (crystals + salt) / max(applied, 1e-9))
    if with_sensor:
        add_sensor_trace(rows, summary, s)
    return rows, summary


def add_sensor_trace(rows, summary, s):
    """Run the probe over the drain trace and let Saltback estimate how much salt washed past it."""
    rng = np.random.default_rng(s["seed"])
    cl = np.array([r["cl_true_mg_L"] for r in rows])
    flow = np.array([r["drain_flow_L_h"] for r in rows])
    wet = ~np.isnan(cl)
    reading, maxed, est, _ = sensor(np.where(wet, cl, s["baseline_cl_mg_L"]), s, rng)
    for r, rd, mx, e, w in zip(rows, reading, maxed, est, wet):
        r.update(reading_mS_cm=rd if w else float("nan"), probe_maxed=bool(mx and w),
                 cl_measured_mg_L=e if w else float("nan"))
    true_load = np.nansum(np.where(wet, cl - s["baseline_cl_mg_L"], 0) * flow) * DT_H      # mg chloride
    seen_load = np.nansum(np.where(wet, np.maximum(est - s["baseline_cl_mg_L"], 0), 0) * flow) * DT_H
    vol = flow[wet].sum() * DT_H
    summary.update(peak_cl_mg_L=float(np.nanmax(cl)) if wet.any() else 0.0,
                   avg_cl_mg_L=float(np.nansum(np.where(wet, cl, 0) * flow) * DT_H / vol) if vol else 0.0,
                   hours_probe_maxed=float((maxed & wet).sum() * DT_H),
                   hours_drain_flowing=float(wet.sum() * DT_H),
                   salt_seen_vs_true_pct=100 * seen_load / true_load if true_load else 100.0)


NEEDED_TOLERANCE_H = 0.25   # "needed" = the smallest rate within 15 minutes of the best icy time any rate reaches
SAFETY_SLACK_H = 0.1        # a recommendation may never add more than 6 minutes of icy road
CANT_FIX_H = 2.0            # still icy this long at the maximum rate: salt alone can't fix this storm
MAX_STEP_DOWN = 0.25        # next-storm advice cuts at most 25% at a time (no cap on safety increases)
MAX_LB = 600                # highest salt rate the search tries, lb per lane-mile per pass
TOO_COLD_F = 15             # state guides advise against more dry rock salt below about 15F
PESSIMISTIC = dict(brine_film_mm=0.06, drain_per_h=0.5, bond_per_h=1.0)   # top of the realistic ranges


def pessimistic(s):
    """The same storm in a worse-than-expected world: brine runs off faster, more is needed, snow bonds faster."""
    return {**s, **{k: max(s[k], v) for k, v in PESSIMISTIC.items()}}


def find_needed(s, max_lb=MAX_LB, step=10):
    """Smallest salt-per-pass that keeps the road within 15 minutes of the safest result any amount reaches.
    Returns (needed_lb_per_pass, sweep) where sweep lists icy hours and chloride for each dose tried."""
    sweep = []
    for lb in range(0, max_lb + 1, step):
        _, sm = storm_timeline(s, lb, with_sensor=False)
        sweep.append(dict(salt_per_pass_lb=lb, total_lb_per_lane_mile=sm["salt_applied_lb_per_lane_mile"],
                          icy_hours=sm["icy_hours"],
                          chloride_released_lb_per_lane_mile=sm["salt_applied_lb_per_lane_mile"] * CL_SHARE))
    best = sweep[-1]["icy_hours"] + NEEDED_TOLERANCE_H
    needed = next(r["salt_per_pass_lb"] for r in sweep if r["icy_hours"] <= best)
    return needed, sweep


def icy_at(s, sweep, lb):
    """Hours icy at a given salt rate (from the sweep when it has that rate)."""
    hit = next((r for r in sweep if r["salt_per_pass_lb"] == lb), None)
    return hit["icy_hours"] if hit else storm_timeline(s, lb, with_sensor=False)[1]["icy_hours"]


def recommend_rate(s, sweep, needed, icy_actual, applied, robust=True):
    """Saltback's advice for one storm, with the safety check built in.
    target = the rate Saltback would have used (need + 10%, raised until it is no icier than the crew's result + 6 min)
    next   = the step to try next storm (cuts capped at 25%; safety increases are not capped)
    cant_fix = True when the road stays icy > 2 h even at the maximum rate: no excess or cut is computed."""
    best = sweep[-1]["icy_hours"]
    if best > CANT_FIX_H:
        return dict(cant_fix=True, target=None, next=None, icy_target=None, icy_next=None, best_icy=best)
    limit = icy_actual + SAFETY_SLACK_H
    target = min(MAX_LB, math.ceil(needed * 1.1 / 10) * 10)
    while target < MAX_LB and icy_at(s, sweep, target) > limit:
        target += 10
    if icy_at(s, sweep, target) > limit and applied > 0:
        target = applied                                   # nothing safer found: keep the crew's rate
    if robust and target < applied:                        # must also be no icier than the crew in the pessimistic world
        ps = pessimistic(s)
        limit_p = storm_timeline(ps, applied, with_sensor=False)[1]["icy_hours"] + SAFETY_SLACK_H
        while target < applied and storm_timeline(ps, target, with_sensor=False)[1]["icy_hours"] > limit_p:
            target += 10
        target = min(target, applied)
    nxt = max(target, math.ceil(applied * (1 - MAX_STEP_DOWN) / 10) * 10) if target < applied else target
    return dict(cant_fix=False, target=target, next=nxt, icy_target=icy_at(s, sweep, target),
                icy_next=icy_at(s, sweep, nxt), best_icy=best)


def storm_card(s, summary, needed_lb, sweep):
    """Saltback's verdict for the storm: the four items in the pitch plus the safety check."""
    applied = summary["salt_per_pass_lb"]
    rec = recommend_rate(s, sweep, needed_lb, summary["icy_hours"], applied)
    route = summary["salt_applied_lb_per_lane_mile"] * s["route_lane_miles"]
    too_cold = s["pavement_temp_F"] < TOO_COLD_F and (rec["cant_fix"] or rec["target"] > applied)
    if too_cold:                                           # more rock salt is the wrong tool here
        rec.update(target=None, next=None, icy_target=None, icy_next=None)
        excess, status = None, "too_cold"
    elif rec["cant_fix"]:
        excess, status = None, "cant_fix"
    else:
        excess = max(0.0, 100 * (applied - rec["target"]) / applied) if applied else 0.0
        status = "under" if rec["target"] > applied else "over" if excess >= 10 else "ok"
    change = lambda r: round(100 * (r - applied) / applied / 5) * 5 if applied and r is not None else None
    return dict(status=status, route_salt_lb=route, needed_per_pass_lb=needed_lb,
                target_per_pass_lb=rec["target"], next_per_pass_lb=rec["next"], excess_pct=excess,
                target_change_pct=change(rec["target"]), next_change_pct=change(rec["next"]),
                icy_actual_h=summary["icy_hours"], icy_target_h=rec["icy_target"], icy_next_h=rec["icy_next"],
                best_icy_h=rec["best_icy"], chloride_released_lb=route * CL_SHARE,
                chloride_to_drain_lb=route * summary["salt_to_drain_pct"] / 100 * CL_SHARE,
                peak_x_stream_limit=summary.get("peak_cl_mg_L", 0) / CL_LIMITS[1])
''')
open('simulate.py', 'w').write(r'''"""Run the Saltback simulation (Steps 1 and 2) and save raw data to results/. (explain.py turns it into plain English.)

Usage:  python simulate.py              uses model.SETTINGS
        python simulate.py my.json      overrides any settings listed in my.json
"""
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import model

OUT = Path("results")

# Widely reprinted chart: pounds of ice melted by 1 lb of salt (originally from the Salt Institute).
# Used ONLY to check the model; the model itself never reads it.
PUBLISHED_ICE_PER_LB = {30: 46.3, 25: 14.4, 20: 8.6, 15: 6.3, 10: 4.9, 5: 4.1, 0: 3.7}


def run(overrides=None):
    s = {**model.SETTINGS, **(overrides or {})}
    rng = np.random.default_rng(s["seed"])
    OUT.mkdir(exist_ok=True)

    # 1) Dose sweep: the same storm salted from 0 up to 2.5x what it needs (at least 600 lb per lane-mile)
    need = model.storm([0.0], s, rng)["salt_needed_lb_per_lane_mile"][0]
    top = int(np.clip(np.ceil(2.5 * np.nan_to_num(need, posinf=0) / 100) * 100, 600, 2500))
    doses_lb = np.arange(0, top + 1, 20)
    sweep = pd.DataFrame(model.storm(doses_lb / model.LB_PER_LANE_MILE, s, rng))
    sweep.round(2).to_csv(OUT / "dose_sweep.csv", index=False)

    # 2) Temperature chart: how much salt this storm needs at each road temperature
    temps = np.arange(-5, 33, 1)
    rows = []
    for t in temps:
        r = model.storm([0.0], {**s, "pavement_temp_F": t}, rng)
        rows.append(dict(pavement_temp_F=t,
                         salt_needed_lb_per_lane_mile=r["salt_needed_lb_per_lane_mile"][0],
                         lb_ice_melted_per_lb_salt=float(model.ice_melted_per_salt(model.f_to_c(t)))))
    pd.DataFrame(rows).round(2).to_csv(OUT / "temperature_chart.csv", index=False)

    # 3) Model check against the published chart
    check = pd.DataFrame([dict(temp_F=t, published=p,
                               model=round(float(model.ice_melted_per_salt(model.f_to_c(t))), 1))
                          for t, p in PUBLISHED_ICE_PER_LB.items()])
    check["difference_pct"] = (100 * (check.model - check.published) / check.published).round(0)
    check.to_csv(OUT / "model_check.csv", index=False)

    # 4) Step 2: the same storm over time (5-minute steps), plus the salt dose it really needed
    rows, summary = model.storm_timeline(s)
    needed, sweep2 = model.find_needed(s)
    card = model.storm_card(s, summary, needed, sweep2)
    pd.DataFrame(rows).round(4).to_csv(OUT / "timeline.csv", index=False)
    pd.DataFrame(sweep2).round(3).to_csv(OUT / "timeline_dose_sweep.csv", index=False)
    (OUT / "storm_summary.json").write_text(json.dumps(dict(summary=summary, card=card), indent=2, default=float))

    (OUT / "settings_used.json").write_text(json.dumps(s, indent=2))
    print(f"Simulated {len(sweep)} salt doses at {s['pavement_temp_F']}F -> {OUT}/dose_sweep.csv, "
          "temperature_chart.csv, model_check.csv,\n  "
          f"and a {len(rows) * model.DT_H:.0f}-hour storm timeline -> timeline.csv, timeline_dose_sweep.csv, storm_summary.json")
    return sweep


if __name__ == "__main__":
    run(json.loads(Path(sys.argv[1]).read_text()) if len(sys.argv) > 1 else None)
''')
open('explain.py', 'w').write(r'''"""Turn the raw numbers in results/ into plain English (report.txt) and charts (chart.png, chart_step2.png).

Run simulate.py first. This file only READS the data; it never re-runs the physics.
"""
import json
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = Path("results")
INK, INK2, GRID, BG = "#0b0b0b", "#52514e", "#e5e4df", "#fcfcfb"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"


def closest(df, col, value):
    return df.iloc[(df[col] - value).abs().argmin()]


def write_report(sweep, temps, check, s):
    need = sweep.salt_needed_lb_per_lane_mile.iloc[0]
    t_c = (s["pavement_temp_F"] - 32) * 5 / 9
    twice = closest(sweep, "salt_applied_lb_per_lane_mile", 2 * need)
    salted = sweep[sweep.salt_applied_lb_per_lane_mile > 0]
    avg_err = (100 * (salted.avg_cl_measured_mg_L - salted.avg_cl_true_mg_L) / salted.avg_cl_true_mg_L).abs().mean()
    no_probe = (100 * (1 - salted.avg_cl_if_no_temp_probe_mg_L / salted.avg_cl_true_mg_L)).mean()
    maxed = int(salted.peak_probe_maxed_out.sum())
    avg_maxed = int(salted.avg_probe_maxed_out.sum())
    seen = (100 * salted.salt_seen_by_sensor_lb_per_lane_mile / salted.salt_reaching_road_lb_per_lane_mile).mean()
    worst_check = check.difference_pct.abs()[check.temp_F <= 25].max()

    L = []
    L += ["SALTBACK SIMULATION - STEP 1: ONE STORM, IN PLAIN ENGLISH", "=" * 60, ""]
    L += [f"THE STORM: road surface at {s['pavement_temp_F']}F ({t_c:.0f}C), with a {s['ice_film_mm']} mm ice film",
          f"stuck to the road after plowing. {s['scatter_loss']:.0%} of the salt bounces off the road.", ""]
    L += ["1) HOW MUCH SALT THIS STORM NEEDED",
          f"   About {need:.0f} lb per lane-mile. Any less leaves some ice; any more melts nothing extra.", ""]
    L += ["   Salt spread      Ice cleared   Salt wasted   Money wasted (per lane-mile)"]
    top = sweep.salt_applied_lb_per_lane_mile.max()
    step = 100 * -(-top // 800)          # about 6-8 table rows, in round 100s
    for dose in range(int(step), int(top) + 1, int(step)):
        r = closest(sweep, "salt_applied_lb_per_lane_mile", dose)
        L.append(f"   {r.salt_applied_lb_per_lane_mile:5.0f} lb/lane-mi   {r.ice_cleared_pct:5.0f}%       "
                 f"{r.excess_pct:5.0f}%        ${r.wasted_dollars_per_lane_mile:5.2f}")
    if need == 0:
        L += ["", "   The road is at or above freezing: the ice melts on its own, so every grain is wasted.", ""]
    else:
        L += ["", f"   At double the needed amount (~{twice.salt_applied_lb_per_lane_mile:.0f} lb), "
              f"{twice.excess_pct:.0f}% of the salt does nothing but pollute.", ""]
    if s["pavement_temp_F"] < 15:
        L += ["   Note: below about 15F salt works very slowly, so real crews often add sand or switch",
              "   to stronger chemicals. This model only knows about plain salt (for now).", ""]
    L += ["2) TEMPERATURE CHANGES EVERYTHING"]
    for t in (30, 25, 20, 15, 10):
        r = closest(temps, "pavement_temp_F", t)
        L.append(f"   At {t:2d}F: 1 lb of salt melts {r.lb_ice_melted_per_lb_salt:4.1f} lb of ice -> "
                 f"this storm needs {r.salt_needed_lb_per_lane_mile:4.0f} lb/lane-mile")
    L += ["   Same storm, very different salt needs. A fixed 'one rate for everything' habit wastes salt",
          "   on mild days and under-treats cold ones. That is why Saltback starts from pavement temperature.", ""]
    L += ["3) CAN THE CHEAP SENSOR SEE IT?",
          f"   Storm-average runoff: the probe stayed in range {len(salted) - avg_maxed} of {len(salted)} times, and",
          f"   Saltback's chloride estimate was off by {avg_err:.0f}% on average. It found {seen:.0f}% of the salt",
          "   that actually reached the road.  -> The DFR0300 works IF it measures well-mixed storm water",
          "   (for example, a collection bucket). Step 2 shows what happens when it logs live in the drain.",
          f"   Without the temperature probe, readings would be {no_probe:.0f}% too low (cold water conducts",
          "   less electricity). -> The $5 DS18B20 is essential, not optional.", ""]
    L += ["4) WARNING FOR THE HARDWARE PLAN",
          f"   The first rush of brine off the road maxed out the probe in {maxed} of {len(salted)} runs (it is",
          "   far saltier than the 20 mS/cm limit). So the DFR0300 cannot measure the first flush. Options:",
          "   measure the storm-average water (works, see #3), or add a higher-range (K=10) probe later.", ""]
    L += ["5) IS THE MODEL TRUSTWORTHY?",
          f"   It was checked against the widely published 'ice melted per pound of salt' chart: within",
          f"   {worst_check:.0f}% from 0F to 25F. (Near 30F it reads higher, because tiny temperature changes",
          "   near freezing matter a lot.) The model never reads that chart; it uses freezing-point physics.", ""]
    return L


def step2_report(tl, sw, summ, card, s):
    n_pass = round(summ["salt_applied_lb_per_lane_mile"] / max(summ["salt_per_pass_lb"], 1e-9))
    flowing = tl.dropna(subset=["cl_true_mg_L"])
    L = ["", "", "STEP 2: THE SAME KIND OF STORM, HOUR BY HOUR", "=" * 60, ""]
    L += [f"THE STORM: {s['snowfall_in']} in of snow over {s['storm_hours']} h with the road at {s['pavement_temp_F']}F.",
          f"The first truck arrives after {s['first_pass_h']} h, then returns every {s['plow_every_h']} h:",
          f"{n_pass} passes x {summ['salt_per_pass_lb']:.0f} lb = {summ['salt_applied_lb_per_lane_mile']:.0f} lb per lane-mile.",
          f"Afterwards the road warms to {s['temp_after_F']}F and the leftover salt washes away.", ""]
    if card["status"] == "too_cold":
        verdict = (f"Too cold for rock salt to work well ({s['pavement_temp_F']}F). State guides recommend sand or a stronger"
                   " chemical such as calcium chloride below about 15F, not more rock salt.")
        card_line = f"   Salt spread: {card['route_salt_lb']:,.0f} lb | Too cold for rock salt"
    elif card["status"] == "cant_fix":
        verdict = ("Salt alone couldn't keep this road clear (still icy "
                   f"{card['best_icy_h']:.1f} h at the maximum rate). Pre-treat with brine, time a pass just before the freeze,"
                   " or use sand or a stronger chemical when it is very cold.")
        card_line = f"   Salt spread: {card['route_salt_lb']:,.0f} lb | Salt alone couldn't fix this storm"
    else:
        verdict = (f"Icy time: {card['icy_actual_h'] * 60:.0f} min as salted -> {card['icy_target_h'] * 60:.0f} min at "
                   f"Saltback's rate ({card['target_per_pass_lb']} lb per pass).")
        card_line = (f"   Salt spread: {card['route_salt_lb']:,.0f} lb | "
                     + ("About the right amount | " if card["status"] == "ok" else f"Excess: {card['excess_pct']:.0f}% | " if card["status"] == "over" else "Under-salted | ") +
                     f"Chloride released: {card['chloride_released_lb']:,.0f} lb | "
                     f"Next storm: try {card['next_per_pass_lb']} lb per pass ({card['next_change_pct']:+.0f}%)")
    L += ["SALTBACK STORM CARD", card_line, ""]
    L += ["6) WAS THE ROAD SAFE?",
          f"   Icy (bonded ice on a freezing road) for {summ['icy_hours']:.1f} h. With no salt: "
          f"{sw.icy_hours.iloc[0]:.1f} h.",
          f"   {verdict}",
          "   Saltback never recommends a rate that would leave the road more than 6 minutes icier than it was,",
          "   either with its best-estimate physics or in a pessimistic case (brine runs off faster, more brine is",
          "   needed, snow bonds faster). It cuts at most 25% per storm, checking the result each time.", ""]
    L += ["7) WHERE THE SALT WENT",
          f"   {summ['salt_to_drain_pct']:.0f}% washed into the storm drain, {summ['salt_to_roadside_pct']:.0f}% was "
          "thrown onto the shoulder and soil",
          "   (bounce, tires, wind). None of it disappears: it all ends up in streams or groundwater.", ""]
    L += ["8) WHAT LEFT THROUGH THE DRAIN",
          f"   The drain ran for {summ['hours_drain_flowing']:.0f} h. Peak chloride {summ['peak_cl_mg_L']:,.0f} mg/L, storm average "
          f"{summ['avg_cl_mg_L']:,.0f} mg/L.",
          f"   Drain water peaked at {card['peak_x_stream_limit']:.0f}x the EPA short-term stream limit (860 mg/L) before dilution.",
          "   A stream dilutes drain water, but chloride never breaks down, so what matters is the total load:",
          f"   {card['chloride_released_lb']:,.0f} lb of chloride from this storm.", ""]
    L += ["9) DID THE SENSOR KEEP UP?",
          f"   The probe was maxed out for {summ['hours_probe_maxed']:.1f} h, so Saltback only saw "
          f"{summ['salt_seen_vs_true_pct']:.0f}% of the salt."]
    if summ["hours_probe_maxed"] > 0.5:
        L += ["   The brine coming off a freezing road is far too salty for the DFR0300 (20 mS/cm max).",
              "   A high-range K=10 probe fixes this: set probe_max_mS_cm to 200 and run again."]
    else:
        L += ["   The probe stayed in range: the drain measurement is trustworthy for this storm."]
    L += ["", "NOT IN THE MODEL YET (coming in later steps):",
          "   - real lab data from Experiments 1 and 2 (the season analysis and uncertainty tests are on the website)",
          "", "All inputs are typical published values, not real town data. See results/settings_used.json."]
    return L


def style(ax, title, xlabel, ylabel):
    ax.set_facecolor(BG)
    ax.set_title(title, loc="left", fontsize=11, color=INK, fontweight="bold")
    ax.set_xlabel(xlabel, color=INK2, fontsize=9)
    ax.set_ylabel(ylabel, color=INK2, fontsize=9)
    ax.grid(color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK2, labelsize=8)


def draw_chart(sweep, temps, s):
    need = sweep.salt_needed_lb_per_lane_mile.iloc[0]
    x = sweep.salt_applied_lb_per_lane_mile
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), facecolor=BG)

    ax = axes[0]
    style(ax, "A. More salt stops helping", "Salt spread (lb per lane-mile)", "Ice cleared (%)")
    ax.axvspan(need, x.max(), color=ORANGE, alpha=0.10, linewidth=0)
    ax.plot(x, sweep.ice_cleared_pct, color=BLUE, linewidth=2)
    ax.axvline(need, color=INK2, linewidth=1, linestyle="--")
    ax.text(need + 10, 45, f"Needed:\n{need:.0f} lb", color=INK, fontsize=9)
    ax.text(x.max() - 10, 8, "Wasted salt zone", color=INK2, fontsize=9, ha="right")
    ax.set_ylim(0, 105)

    ax = axes[1]
    style(ax, "B. Colder road, more salt needed", "Pavement temperature (F)", "Salt needed (lb per lane-mile)")
    ax.plot(temps.pavement_temp_F, temps.salt_needed_lb_per_lane_mile, color=BLUE, linewidth=2)
    ax.plot([s["pavement_temp_F"]], [need], "o", color=BLUE, markersize=8, markeredgecolor=BG, markeredgewidth=2)
    ax.annotate(f"This storm ({s['pavement_temp_F']}F)", (s["pavement_temp_F"], need),
                xytext=(8, 10), textcoords="offset points", color=INK, fontsize=9)

    ax = axes[2]
    style(ax, "C. What the drain sensor reports", "Salt spread (lb per lane-mile)", "Storm-average chloride (mg/L)")
    for col, color, label in [("avg_cl_true_mg_L", INK2, "True"),
                              ("avg_cl_measured_mg_L", BLUE, "Saltback estimate"),
                              ("avg_cl_if_no_temp_probe_mg_L", ORANGE, "No temperature probe")]:
        ls = "--" if label == "True" else "-"
        ax.plot(x, sweep[col], color=color, linewidth=2, linestyle=ls, label=label)
        ax.text(x.iloc[-1] + 8, sweep[col].iloc[-1], label, color=INK, fontsize=8, va="center")
    ax.legend(frameon=False, fontsize=8, loc="upper left", labelcolor=INK)
    ax.set_xlim(0, x.max() * 1.33)

    fig.tight_layout()
    fig.savefig(OUT / "chart.png", dpi=150, facecolor=BG)
    plt.close(fig)


def draw_step2_chart(tl, sw, card, s):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), facecolor=BG)
    storm_end = s["storm_hours"]

    ax = axes[0]
    style(ax, "D. Ice bonded to the road", "Hours since snow started", "Bonded ice (mm)")
    ax.axvspan(0, storm_end, color=BLUE, alpha=0.06, linewidth=0)
    ax.plot(tl.hour, tl.bonded_ice_mm, color=BLUE, linewidth=2)
    ax.axhline(0.05, color=ORANGE, linewidth=1, linestyle="--")
    ax.text(tl.hour.max(), 0.05, "  icy above this", color=INK2, fontsize=8, va="bottom", ha="right")
    ax.text(storm_end / 2, ax.get_ylim()[1] * 0.97, "snowing", color=INK2, fontsize=8, ha="center", va="top")

    ax = axes[1]
    style(ax, "E. Chloride in the drain (log scale)", "Hours since snow started", "Chloride (mg/L)")
    ax.set_yscale("log")
    ax.plot(tl.hour, tl.cl_true_mg_L, color=INK2, linewidth=2, linestyle="--", label="True")
    ax.plot(tl.hour, tl.cl_measured_mg_L, color=BLUE, linewidth=2, label="Sensor + Saltback")
    ax.axhline(860, color=ORANGE, linewidth=1)
    ax.text(0.5, 860 * 0.93, "EPA short-term limit", color=INK2, fontsize=8, va="top")
    ax.legend(frameon=False, fontsize=8, loc="upper right", labelcolor=INK)

    ax = axes[2]
    style(ax, "F. Salt per pass vs. icy hours", "Salt per pass (lb per lane-mile)", "Hours the road was icy")
    ax.plot(sw.salt_per_pass_lb, sw.icy_hours, color=BLUE, linewidth=2)
    marks = [(s["salt_per_pass_lb"], "used", (10, -18))]
    if card["target_per_pass_lb"] is not None:
        marks.insert(0, (card["target_per_pass_lb"], "Saltback", (10, 22)))
    for lb, label, off in marks:
        r = closest(sw, "salt_per_pass_lb", lb)
        ax.plot([r.salt_per_pass_lb], [r.icy_hours], "o", color=BLUE if label == "Saltback" else ORANGE,
                markersize=8, markeredgecolor=BG, markeredgewidth=2)
        ax.annotate(f"{label}: {r.salt_per_pass_lb:.0f} lb", (r.salt_per_pass_lb, r.icy_hours),
                    xytext=off, textcoords="offset points", color=INK, fontsize=9)

    fig.tight_layout()
    fig.savefig(OUT / "chart_step2.png", dpi=150, facecolor=BG)
    plt.close(fig)


def run():
    sweep = pd.read_csv(OUT / "dose_sweep.csv")
    temps = pd.read_csv(OUT / "temperature_chart.csv")
    check = pd.read_csv(OUT / "model_check.csv")
    tl = pd.read_csv(OUT / "timeline.csv")
    sw = pd.read_csv(OUT / "timeline_dose_sweep.csv")
    st = json.loads((OUT / "storm_summary.json").read_text())
    s = json.loads((OUT / "settings_used.json").read_text())
    text = "\n".join(write_report(sweep, temps, check, s) + step2_report(tl, sw, st["summary"], st["card"], s))
    (OUT / "report.txt").write_text(text)
    print(text)
    draw_chart(sweep, temps, s)
    draw_step2_chart(tl, sw, st["card"], s)
    print(f"\nCharts saved to {OUT}/chart.png and {OUT}/chart_step2.png")


if __name__ == "__main__":
    run()
''')
print('Model files ready: model.py, simulate.py, explain.py')

In [ ]:
# @title 2. Storm settings - drag the sliders, then run this cell and the ones below
import json
pavement_temp_F = 22     # @param {type:"slider", min:-5, max:32, step:1}
ice_film_mm = 0.2        # @param {type:"slider", min:0.05, max:0.5, step:0.05}
scatter_loss = 0.2       # @param {type:"slider", min:0.0, max:0.4, step:0.05}
later_water_mm = 5.0     # @param {type:"slider", min:0, max:20, step:1}
unsalted_share = 0.3     # @param {type:"slider", min:0.0, max:0.7, step:0.05}
water_temp_C = 1.0       # @param {type:"slider", min:0, max:5, step:0.5}
salt_price_per_ton = 90  # @param {type:"slider", min:60, max:130, step:5}
# --- Step 2: the storm over time ---
snowfall_in = 4          # @param {type:"slider", min:0, max:18, step:0.5}
storm_hours = 8          # @param {type:"slider", min:4, max:24, step:1}
salt_per_pass_lb = 220   # @param {type:"slider", min:0, max:500, step:10}
first_pass_h = 0.5       # @param {type:"slider", min:0, max:3, step:0.25}
plow_every_h = 2         # @param {type:"slider", min:1, max:4, step:0.5}
temp_after_F = 38        # @param {type:"slider", min:10, max:50, step:1}
probe = "DFR0300 (max 20 mS/cm)"  # @param ["DFR0300 (max 20 mS/cm)", "High-range K=10 (max 200 mS/cm)"]
settings = dict(pavement_temp_F=pavement_temp_F, ice_film_mm=ice_film_mm, scatter_loss=scatter_loss,
                later_water_mm=later_water_mm, unsalted_share=unsalted_share,
                water_temp_C=water_temp_C, salt_price_per_ton=salt_price_per_ton,
                snowfall_in=snowfall_in, storm_hours=storm_hours, salt_per_pass_lb=salt_per_pass_lb,
                first_pass_h=first_pass_h, plow_every_h=plow_every_h, temp_after_F=temp_after_F,
                probe_max_mS_cm=20.0 if probe.startswith("DFR") else 200.0)
json.dump(settings, open('settings.json', 'w'))
print('Settings saved:', settings)

In [ ]:
# @title 3. Run the simulation (raw data)
!python simulate.py settings.json
import pandas as pd
pd.read_csv('results/dose_sweep.csv').head(10)

In [ ]:
# @title 4. Translate into plain English + chart
!python explain.py
from IPython.display import Image, display
display(Image('results/chart.png'))
display(Image('results/chart_step2.png'))

In [ ]:
# @title 5. (Optional) Download all results as a zip
!zip -qr saltback_results.zip results
from google.colab import files
files.download('saltback_results.zip')